# 🍔 Swiggy Food Delivery Analysis

**Tools:** Python · Pandas · Seaborn · Matplotlib  
**Dataset:** Swiggy Restaurant Data (8681 rows, 6+ cities)  
**Columns:** ID, Area, City, Restaurant, Price, Avg ratings, Total ratings, Cuisine, Address, Delivery time

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0E1117',
    'axes.facecolor': '#1A1F2E',
    'axes.edgecolor': '#2A3040',
    'axes.labelcolor': '#CCCCCC',
    'xtick.color': '#AAAAAA',
    'ytick.color': '#AAAAAA',
    'text.color': '#EEEEEE',
    'grid.color': '#2A3040',
    'grid.alpha': 0.4,
    'font.size': 12,
})

PALETTE = ['#FF6B6B', '#FF8E53', '#FFC857', '#4ECDC4', '#45B7D1',
           '#96E6A1', '#DDA0DD', '#F0E68C', '#87CEEB', '#FFB6C1',
           '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9', '#F1948A']

print('✅ Libraries loaded successfully!')

In [ ]:
# Load dataset
import sys
sys.path.insert(0, '..')
from utils import load_data, explode_cuisines

df = load_data('../data/swiggy.csv')
print(f'Dataset shape: {df.shape}')
print(f'Cities: {df["City"].nunique()} — {df["City"].unique().tolist()}')
df.head()

In [ ]:
# Basic info
print('\n📊 Dataset Info:')
print(f'Total Restaurants: {len(df):,}')
print(f'Avg Rating: {df["Rating"].mean():.2f}')
print(f'Avg Cost for Two: ₹{df["Cost_for_Two"].mean():.0f}')
print(f'Avg Delivery Time: {df["Delivery_Time"].mean():.1f} min')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nData Types:\n{df.dtypes}')

In [ ]:
# Exploded cuisine dataframe
df_exp = explode_cuisines(df)
print(f'Exploded dataset shape: {df_exp.shape}')
print(f'Unique cuisines: {df_exp["Cuisine"].nunique()}')
df_exp.head()

---
## 2. ⭐ Core Insights

### 2.1 Most Popular Cuisines (by Count & Ratings)

In [ ]:
# Top 15 most popular cuisines by restaurant count
top_cuisines = (
    df_exp.groupby('Cuisine')
    .agg(Count=('Restaurant', 'count'), Avg_Rating=('Rating', 'mean'))
    .reset_index()
    .sort_values('Count', ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# By count
axes[0].barh(top_cuisines['Cuisine'].values[::-1], top_cuisines['Count'].values[::-1],
             color=PALETTE[:15][::-1], edgecolor='#0E1117', height=0.7)
for i, v in enumerate(top_cuisines['Count'].values[::-1]):
    axes[0].text(v + 10, i, str(v), va='center', fontsize=10, color='#EEE')
axes[0].set_xlabel('Number of Restaurants')
axes[0].set_title('Top 15 Cuisines by Count', fontsize=14)

# By rating
axes[1].barh(top_cuisines['Cuisine'].values[::-1], top_cuisines['Avg_Rating'].values[::-1],
             color=PALETTE[:15][::-1], edgecolor='#0E1117', height=0.7)
for i, v in enumerate(top_cuisines['Avg_Rating'].values[::-1]):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=10, color='#EEE')
axes[1].set_xlabel('Average Rating')
axes[1].set_xlim(left=3.5)
axes[1].set_title('Avg Rating of Top 15 Cuisines', fontsize=14)

plt.tight_layout()
plt.savefig('../assets/top_cuisines.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Best-Rated Cuisines (min 100+ restaurants)

In [ ]:
cuisine_stats = (
    df_exp.groupby('Cuisine')
    .agg(Count=('Restaurant', 'count'), Avg_Rating=('Rating', 'mean'))
    .reset_index()
)
best_rated = cuisine_stats[cuisine_stats['Count'] >= 100].sort_values('Avg_Rating', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(best_rated['Cuisine'].values[::-1], best_rated['Avg_Rating'].values[::-1],
               color=PALETTE[:len(best_rated)][::-1], edgecolor='#0E1117', height=0.7)
for bar in bars:
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.2f}', va='center', fontsize=11, color='#EEE')
ax.set_xlabel('Average Rating', fontsize=13)
ax.set_xlim(left=3.8)
ax.set_title('Best-Rated Cuisines (100+ restaurants)', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/best_rated_cuisines.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Best Rated Cuisines:')
best_rated[['Cuisine', 'Count', 'Avg_Rating']].to_string(index=False)

### 2.3 Locations with Highest Restaurant Density

In [ ]:
area_density = (
    df.groupby('Area')
    .agg(Restaurant_Count=('Restaurant', 'count'),
         Avg_Rating=('Rating', 'mean'),
         Avg_Cost=('Cost_for_Two', 'mean'))
    .reset_index()
    .sort_values('Restaurant_Count', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(range(len(area_density)), area_density['Restaurant_Count'].values,
              color=PALETTE[:len(area_density)], edgecolor='#0E1117')
ax.set_xticks(range(len(area_density)))
ax.set_xticklabels(area_density['Area'].values, rotation=45, ha='right', fontsize=9)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{int(bar.get_height())}', ha='center', fontsize=9, color='#EEE')
ax.set_ylabel('Number of Restaurants', fontsize=13)
ax.set_title('Top 20 Areas by Restaurant Density', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/restaurant_density.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Cost vs Rating Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Scatter plot
scatter = axes[0].scatter(df['Cost_for_Two'], df['Rating'],
                          alpha=0.3, s=15, c=df['Delivery_Time'],
                          cmap='coolwarm', edgecolors='none')
# Trend line
z = np.polyfit(df['Cost_for_Two'].dropna(), df.loc[df['Cost_for_Two'].notna(), 'Rating'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Cost_for_Two'].min(), df['Cost_for_Two'].max(), 100)
axes[0].plot(x_line, p(x_line), '--', color='#FF6B6B', linewidth=2, label='Trend')
plt.colorbar(scatter, ax=axes[0], shrink=0.8, label='Delivery Time (min)')
axes[0].set_xlabel('Cost for Two (₹)', fontsize=13)
axes[0].set_ylabel('Rating', fontsize=13)
axes[0].set_title('Cost vs Rating', fontsize=15)
axes[0].legend()

# Correlation heatmap
corr_cols = ['Cost_for_Two', 'Rating', 'Votes', 'Delivery_Time']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, linecolor='#1A1F2E', ax=axes[1],
            vmin=-1, vmax=1, center=0)
axes[1].set_title('Correlation Heatmap', fontsize=15)

plt.tight_layout()
plt.savefig('../assets/cost_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

corr = df['Cost_for_Two'].corr(df['Rating'])
print(f'\n📊 Cost vs Rating Correlation: {corr:.4f}')
if corr > 0.1:
    print('→ Positive correlation: Expensive ≈ slightly better rated')
else:
    print('→ Weak/No correlation: Price does NOT guarantee quality!')

### 2.5 Average Delivery Time by Location

In [ ]:
delivery_by_area = (
    df.groupby('Area')['Delivery_Time'].agg(['mean', 'count'])
    .reset_index()
    .query('count >= 10')
    .sort_values('mean')
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Fastest
fastest = delivery_by_area.head(15)
axes[0].barh(fastest['Area'].values[::-1], fastest['mean'].values[::-1],
             color='#4ECDC4', edgecolor='#0E1117', height=0.65)
for i, v in enumerate(fastest['mean'].values[::-1]):
    axes[0].text(v + 0.3, i, f'{v:.1f} min', va='center', fontsize=10, color='#EEE')
axes[0].set_xlabel('Avg Delivery Time (min)', fontsize=13)
axes[0].set_title('🚀 Fastest Delivery Areas', fontsize=14)

# Slowest
slowest = delivery_by_area.tail(15)
axes[1].barh(slowest['Area'].values, slowest['mean'].values,
             color='#FF6B6B', edgecolor='#0E1117', height=0.65)
for i, v in enumerate(slowest['mean'].values):
    axes[1].text(v + 0.3, i, f'{v:.1f} min', va='center', fontsize=10, color='#EEE')
axes[1].set_xlabel('Avg Delivery Time (min)', fontsize=13)
axes[1].set_title('🐌 Slowest Delivery Areas', fontsize=14)

plt.tight_layout()
plt.savefig('../assets/delivery_time_areas.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. 🥇 Customer Value Analysis

### 3.1 Best Value-for-Money Restaurants

In [ ]:
from utils import get_value_for_money, get_budget_friendly

vfm = get_value_for_money(df, top_n=15)
print('🏆 Best Value for Money Restaurants (High Rating + Low Cost):\n')
vfm[['Restaurant', 'Area', 'City', 'Rating', 'Cost_for_Two', 'Cuisine', 'Value_Score']].head(15)

### 3.2 Top 10 Budget-Friendly Restaurants

In [ ]:
budget = get_budget_friendly(df, max_cost=300, min_rating=4.0, top_n=10)
print('💸 Top 10 Budget-Friendly Restaurants (≤₹300, Rating ≥ 4.0):\n')

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(budget['Restaurant'].values[::-1], budget['Rating'].values[::-1],
               color='#96E6A1', edgecolor='#0E1117', height=0.65)
for bar, cost in zip(bars, budget['Cost_for_Two'].values[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f} ⭐ (₹{int(cost)})', va='center', fontsize=10, color='#EEE')
ax.set_xlabel('Rating', fontsize=13)
ax.set_xlim(left=4.0)
ax.set_title('Top 10 Budget-Friendly Restaurants', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/budget_friendly.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 📍 Location Intelligence

### 4.1 Cost Variation Across Locations

In [ ]:
top_areas = df['Area'].value_counts().head(10).index.tolist()
df_top = df[df['Area'].isin(top_areas)]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Box plot: Cost variation
sns.boxplot(data=df_top, x='Area', y='Cost_for_Two', palette=PALETTE[:10], ax=axes[0],
            flierprops=dict(marker='o', markerfacecolor='#FF6B6B', markersize=4))
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[0].set_xlabel('')
axes[0].set_ylabel('Cost for Two (₹)', fontsize=13)
axes[0].set_title('Cost Variation Across Top Areas', fontsize=14)

# Violin plot: Rating distribution
sns.violinplot(data=df_top, x='Area', y='Rating', palette=PALETTE[:10], ax=axes[1], inner='quartile')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[1].set_xlabel('')
axes[1].set_ylabel('Rating', fontsize=13)
axes[1].set_title('Rating Distribution per Area', fontsize=14)

plt.tight_layout()
plt.savefig('../assets/location_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Food Hubs (High Density + High Ratings)

In [ ]:
from utils import get_food_hubs

hubs = get_food_hubs(df, min_restaurants=10, min_avg_rating=4.0)
print(f'🏆 Found {len(hubs)} Food Hubs\n')

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    hubs['Restaurant_Count'], hubs['Avg_Rating'],
    s=hubs['Avg_Cost'] * 0.6, c=hubs['Avg_Delivery'],
    cmap='YlOrRd', alpha=0.8, edgecolors='#333', linewidth=1.5,
)
for _, row in hubs.iterrows():
    ax.annotate(row['Area'], (row['Restaurant_Count'], row['Avg_Rating']),
                fontsize=8, color='#EEE', ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
cbar.set_label('Avg Delivery Time (min)', fontsize=11)
ax.set_xlabel('Number of Restaurants', fontsize=13)
ax.set_ylabel('Average Rating', fontsize=13)
ax.set_title('Food Hubs — Bubble Size = Avg Cost, Color = Delivery Time', fontsize=14)
plt.tight_layout()
plt.savefig('../assets/food_hubs.png', dpi=150, bbox_inches='tight')
plt.show()

hubs[['Area', 'Restaurant_Count', 'Avg_Rating', 'Avg_Cost', 'Avg_Delivery']].round(2)

---
## 5. ⏱️ Delivery Performance

### 5.1 Fastest Delivery Cuisines

In [ ]:
from utils import get_delivery_by_cuisine

fast_cuisines = get_delivery_by_cuisine(df_exp, top_n=15)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(fast_cuisines['Cuisine'].values[::-1], fast_cuisines['Avg_Delivery'].values[::-1],
               color=PALETTE[:len(fast_cuisines)][::-1], edgecolor='#0E1117', height=0.65)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f} min', va='center', fontsize=10, color='#EEE')
ax.set_xlabel('Avg Delivery Time (min)', fontsize=13)
ax.set_title('Fastest Delivery Cuisines (50+ restaurants)', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/delivery_cuisines.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Does Faster Delivery = Better Ratings?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=df, x='Delivery_Time', y='Rating', alpha=0.3, s=15,
                color='#45B7D1', ax=ax, edgecolor='none')

# Trend line
z = np.polyfit(df['Delivery_Time'].dropna(), df.loc[df['Delivery_Time'].notna(), 'Rating'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Delivery_Time'].min(), df['Delivery_Time'].max(), 100)
ax.plot(x_line, p(x_line), '--', color='#FF6B6B', linewidth=2, label='Trend')

corr = df['Delivery_Time'].corr(df['Rating'])
ax.set_xlabel('Delivery Time (min)', fontsize=13)
ax.set_ylabel('Rating', fontsize=13)
ax.set_title(f'Delivery Time vs Rating (Correlation: {corr:.3f})', fontsize=15)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../assets/delivery_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Delivery Time vs Rating Correlation: {corr:.4f}')
if abs(corr) < 0.1:
    print('→ No significant correlation — delivery speed alone doesn\'t determine ratings')
elif corr < 0:
    print('→ Slight negative correlation — faster delivery tends to have slightly better ratings')

---
## 6. 🔍 Popularity vs Quality

### 6.1 Overhyped Restaurants (High Votes + Low Rating)

In [ ]:
from utils import get_overhyped, get_hidden_gems

overhyped = get_overhyped(df, min_votes=1000, max_rating=3.5, top_n=15)
print(f'📢 Found {len(overhyped)} Overhyped Restaurants\n')

fig, ax = plt.subplots(figsize=(14, 7))
scatter = ax.scatter(overhyped['Votes'], overhyped['Rating'],
                     s=overhyped['Cost_for_Two'] * 0.4,
                     color='#FF6B6B', alpha=0.7, edgecolors='#333')
for _, row in overhyped.iterrows():
    ax.annotate(row['Restaurant'][:25], (row['Votes'], row['Rating']),
                fontsize=8, color='#EEE', xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Total Votes', fontsize=13)
ax.set_ylabel('Rating', fontsize=13)
ax.set_title('Overhyped — High Votes but Low Ratings', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/overhyped.png', dpi=150, bbox_inches='tight')
plt.show()

overhyped[['Restaurant', 'Area', 'City', 'Rating', 'Votes', 'Cost_for_Two']]

### 6.2 Hidden Gems (Low Votes + High Rating)

In [ ]:
gems = get_hidden_gems(df, max_votes=100, min_rating=4.5, top_n=15)
print(f'💎 Found {len(gems)} Hidden Gems\n')

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(gems['Restaurant'].values[::-1], gems['Rating'].values[::-1],
               color='#96E6A1', edgecolor='#0E1117', height=0.65)
for bar, votes in zip(bars, gems['Votes'].values[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f} ⭐ ({int(votes)} votes)', va='center', fontsize=10, color='#EEE')
ax.set_xlabel('Rating', fontsize=13)
ax.set_xlim(left=4.3)
ax.set_title('💎 Hidden Gems — High Rating, Low Votes', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/hidden_gems.png', dpi=150, bbox_inches='tight')
plt.show()

gems[['Restaurant', 'Area', 'City', 'Rating', 'Votes', 'Cost_for_Two', 'Cuisine']]

---
## 7. 🍽️ Cuisine Trends

### 7.1 Multi-Cuisine vs Single-Cuisine Performance

In [ ]:
cuisine_type_stats = df.groupby('Cuisine_Type').agg(
    Count=('Restaurant', 'count'),
    Avg_Rating=('Rating', 'mean'),
    Avg_Cost=('Cost_for_Two', 'mean'),
    Avg_Delivery=('Delivery_Time', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [('Avg_Rating', 'Average Rating', (3.5, 4.5)),
           ('Avg_Cost', 'Average Cost (₹)', None),
           ('Avg_Delivery', 'Avg Delivery (min)', None)]

for i, (col, title, ylim) in enumerate(metrics):
    bars = axes[i].bar(cuisine_type_stats['Cuisine_Type'], cuisine_type_stats[col],
                       color=['#FF6B6B', '#4ECDC4'], edgecolor='#0E1117', width=0.5)
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{bar.get_height():.2f}', ha='center', fontsize=12, color='#EEE')
    axes[i].set_ylabel(title, fontsize=12)
    axes[i].set_title(title, fontsize=13)
    if ylim:
        axes[i].set_ylim(ylim)

plt.tight_layout()
plt.savefig('../assets/multi_vs_single.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Multi vs Single Cuisine Stats:')
cuisine_type_stats.round(2)

### 7.2 Which Cuisines Dominate Specific Areas

In [ ]:
# Cuisine popularity heatmap across cities
top_10_cuisines = df_exp.groupby('Cuisine').size().nlargest(10).index.tolist()
heatmap_data = df_exp[df_exp['Cuisine'].isin(top_10_cuisines)]
pivot = heatmap_data.pivot_table(index='Cuisine', columns='City', values='Restaurant', aggfunc='count', fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            linecolor='#1A1F2E', ax=ax, cbar_kws={'shrink': 0.8})
ax.set_xlabel('City', fontsize=13)
ax.set_ylabel('Cuisine', fontsize=13)
ax.set_title('Cuisine Popularity Across Cities', fontsize=15)
plt.tight_layout()
plt.savefig('../assets/cuisine_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 💡 Business Insights

### 8.1 Optimal Price Range for Highest Ratings

In [ ]:
from utils import get_optimal_price_range

price_stats = get_optimal_price_range(df)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Rating by price range
x = range(len(price_stats))
bars = axes[0].bar(x, price_stats['Avg_Rating'], color=PALETTE[:len(price_stats)],
                   edgecolor='#0E1117', width=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(price_stats['Price_Range'], rotation=30, ha='right')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.2f}', ha='center', fontsize=11, color='#EEE')
axes[0].set_ylabel('Average Rating', fontsize=13)
axes[0].set_ylim(3.5, 4.5)
axes[0].set_title('Avg Rating by Price Range', fontsize=14)

# Count by price range
bars = axes[1].bar(x, price_stats['Count'], color=PALETTE[:len(price_stats)],
                   edgecolor='#0E1117', width=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(price_stats['Price_Range'], rotation=30, ha='right')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{int(bar.get_height())}', ha='center', fontsize=11, color='#EEE')
axes[1].set_ylabel('Restaurant Count', fontsize=13)
axes[1].set_title('Restaurant Count by Price Range', fontsize=14)

plt.tight_layout()
plt.savefig('../assets/optimal_price.png', dpi=150, bbox_inches='tight')
plt.show()

best = price_stats.loc[price_stats['Avg_Rating'].idxmax()]
print(f'\n🏆 Optimal Price Range: {best["Price_Range"]} (Avg Rating: {best["Avg_Rating"]:.2f})')

### 8.2 Best Location + Cuisine Combo for New Restaurant

In [ ]:
from utils import get_best_location_cuisine_combo

combos = get_best_location_cuisine_combo(df_exp, top_n=15)
print('🏪 Best Location + Cuisine Combos for a New Restaurant:\n')
combos[['Area', 'Cuisine', 'Count', 'Avg_Rating', 'Avg_Cost', 'Score']].round(2)

---
## 📝 Summary of Key Findings

| Insight | Finding |
|---------|--------|
| Most Popular Cuisines | North Indian, Chinese, Fast Food dominate |
| Cost vs Rating | Weak positive correlation — price ≠ quality |
| Best Value | High ratings possible at ₹100-300 range |
| Hidden Gems | Many 4.5+ rated restaurants with <100 votes |
| Delivery | Avg ~55 min, no strong link to ratings |
| Multi vs Single | Multi-cuisine restaurants are far more common |
| Business Tip | Mid-range (₹400-600) gets best ratings |

### 🎯 Recommendations
1. **For customers:** Look for hidden gems — great food, fewer crowds
2. **For restaurants:** Mid-range pricing yields best ratings
3. **For entrepreneurs:** Study food hubs for new restaurant locations
4. **For delivery:** Speed varies significantly by area, not by cuisine